# Clarity Classification Ensemble (PyTorch Lightning + PEFT)

Train a 3-class clarity classifier (Clear Reply / Clear Non-Reply / Ambivalent) on QEvasion with:
- Stratified K-fold ensemble
- Optional PEFT (LoRA)
- Class weights, label smoothing, early stopping

**Requirements:** `pip install pytorch-lightning transformers datasets scikit-learn torchmetrics peft`

In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install pytorch-lightning transformers datasets scikit-learn torchmetrics peft

## 1. Imports

In [ ]:
from __future__ import annotations

import json
import math
import shutil
from collections import Counter
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
from typing import Optional

import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
from datasets import load_dataset
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy, F1Score, Precision, Recall
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

try:
    from peft import LoraConfig, PeftModel, TaskType, get_peft_model
except ImportError:
    LoraConfig = TaskType = get_peft_model = PeftModel = None

print("Imports OK")

## 2. Configuration

Edit the config below (e.g. `use_peft=True`, `accelerator="cpu"` to avoid MPS bugs).

In [ ]:
LABEL2ID = {"Clear Reply": 0, "Clear Non-Reply": 1, "Ambivalent": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 3

CONFIG = {
    "model": "roberta-base",
    "use_peft": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": None,
    "merge_lora": True,
    "merge_lora_output": "",
    "epochs": 5,
    "batch_size": 16,
    "grad_accum": 2,
    "lr": 2e-5,
    "weight_decay": 0.1,
    "warmup_ratio": 0.1,
    "patience": 2,
    "dropout": 0.2,
    "label_smoothing": 0.1,
    "max_length": 256,
    "num_workers": 0,
    "num_folds": 3,
    "output_dir": "./clarity_ensemble",
    "seed": 42,
    "overwrite_ensemble_dirs": True,
    "overwrite_ensemble_root": ".",
    "accelerator": "cpu",
    "devices": 1,
    "precision": "32-true",
}

args = SimpleNamespace(**CONFIG)
print(json.dumps(CONFIG, indent=2))

## 3. Dataset

In [ ]:
class ClarityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }

print("ClarityDataset defined")

## 4. Lightning Module (ClarityClassifier)

In [ ]:
class ClarityClassifier(pl.LightningModule):
    def __init__(
        self,
        model_name="roberta-base",
        num_classes=3,
        learning_rate=2e-5,
        weight_decay=0.1,
        warmup_ratio=0.1,
        dropout=0.2,
        label_smoothing=0.1,
        class_weights=None,
        total_steps=1000,
        use_peft=False,
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        lora_target_modules=None,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["class_weights"])

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
            classifier_dropout=dropout,
        )
        if use_peft and get_peft_model:
            target_modules = lora_target_modules or ["query", "key", "value"]
            lora_config = LoraConfig(
                task_type=TaskType.SEQ_CLS,
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                target_modules=target_modules,
            )
            self.model = get_peft_model(self.model, lora_config)

        self.class_weights = class_weights
        self.label_smoothing = label_smoothing

        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.train_f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")
        self.val_precision = Precision(task="multiclass", num_classes=num_classes, average="macro")
        self.val_recall = Recall(task="multiclass", num_classes=num_classes, average="macro")
        self.val_f1_per_class = F1Score(task="multiclass", num_classes=num_classes, average=None)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")
        self.test_precision = Precision(task="multiclass", num_classes=num_classes, average="macro")
        self.test_recall = Recall(task="multiclass", num_classes=num_classes, average="macro")

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

    def _compute_loss(self, logits, labels):
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        return F.cross_entropy(logits, labels, weight=weights, label_smoothing=self.label_smoothing)

    def training_step(self, batch, batch_idx):
        outputs = self(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        loss = self._compute_loss(outputs.logits, batch["label"])
        preds = torch.argmax(outputs.logits, dim=-1)
        self.train_acc(preds, batch["label"])
        self.train_f1(preds, batch["label"])
        self.log("train/loss", loss, prog_bar=True)
        self.log("train/acc", self.train_acc, prog_bar=True)
        self.log("train/f1_macro", self.train_f1, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        loss = self._compute_loss(outputs.logits, batch["label"])
        preds = torch.argmax(outputs.logits, dim=-1)
        self.val_acc(preds, batch["label"])
        self.val_f1(preds, batch["label"])
        self.val_precision(preds, batch["label"])
        self.val_recall(preds, batch["label"])
        self.val_f1_per_class(preds, batch["label"])
        self.log("val/loss", loss, prog_bar=True)
        self.log("val/acc", self.val_acc, prog_bar=True)
        self.log("val/f1_macro", self.val_f1, prog_bar=True)
        self.log("val/precision", self.val_precision)
        self.log("val/recall", self.val_recall)
        return loss

    def test_step(self, batch, batch_idx):
        outputs = self(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        loss = self._compute_loss(outputs.logits, batch["label"])
        preds = torch.argmax(outputs.logits, dim=-1)
        self.test_acc(preds, batch["label"])
        self.test_f1(preds, batch["label"])
        self.test_precision(preds, batch["label"])
        self.test_recall(preds, batch["label"])
        self.log("test/loss", loss, prog_bar=True)
        self.log("test/acc", self.test_acc, prog_bar=True)
        self.log("test/f1_macro", self.test_f1, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        f1_per_class = self.val_f1_per_class.compute()
        for i, f1 in enumerate(f1_per_class):
            self.log(f"val/f1_{ID2LABEL[i]}", f1)
        self.val_f1_per_class.reset()
        return

    def configure_optimizers(self):
        no_decay = ["bias", "LayerNorm.weight", "layernorm.weight"]
        optimizer_grouped_parameters = [
            {"params": [p for n, p in self.model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": self.hparams.weight_decay},
            {"params": [p for n, p in self.model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
        ]
        optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=self.hparams.learning_rate, eps=1e-8)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(self.hparams.total_steps * self.hparams.warmup_ratio),
            num_training_steps=self.hparams.total_steps,
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "step", "frequency": 1}}

print("ClarityClassifier defined")

## 5. Data Module

In [ ]:
class ClarityDataModule(pl.LightningDataModule):
    def __init__(self, model_name, batch_size, max_length, num_workers,
                 train_texts, train_labels, val_texts, val_labels, test_texts, test_labels):
        super().__init__()
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length
        self.num_workers = num_workers
        self.train_texts, self.train_labels = train_texts, train_labels
        self.val_texts, self.val_labels = val_texts, val_labels
        self.test_texts, self.test_labels = test_texts, test_labels
        self.tokenizer = None

    def setup(self, stage=None):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if stage in ("fit", None):
            self.train_dataset = ClarityDataset(self.train_texts, self.train_labels, self.tokenizer, self.max_length)
            self.val_dataset = ClarityDataset(self.val_texts, self.val_labels, self.tokenizer, self.max_length)
        if stage in ("test", None) and self.test_texts is not None:
            self.test_dataset = ClarityDataset(self.test_texts, self.test_labels, self.tokenizer, self.max_length)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True)
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True)
    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True)

print("ClarityDataModule defined")

## 6. Data loading & training helpers

In [ ]:
def load_qevasion_data():
    dataset = load_dataset("ailsntua/QEvasion")
    def prepare(split):
        ds = dataset[split]
        texts = [f"Question: {q}\nAnswer: {a}" for q, a in zip(ds["question"], ds["interview_answer"])]
        labels = [LABEL2ID.get(lbl, 2) for lbl in ds["clarity_label"]]
        return texts, labels
    train_texts, train_labels = prepare("train")
    test_texts, test_labels = prepare("test")
    print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")
    print(f"Train labels: {Counter(train_labels)}")
    return train_texts, train_labels, test_texts, test_labels

def compute_class_weights_tensor(labels):
    classes = np.array(sorted(set(labels)))
    w = compute_class_weight("balanced", classes=classes, y=np.array(labels))
    return torch.tensor(w, dtype=torch.float32)

print("Helpers defined")

## 7. Train single fold & full ensemble

In [ ]:
def train_single_fold(fold_idx, train_texts, train_labels, val_texts, val_labels, test_texts, test_labels, args, output_dir):
    class_weights = compute_class_weights_tensor(train_labels)
    dm = ClarityDataModule(
        args.model, args.batch_size, args.max_length, args.num_workers,
        train_texts, train_labels, val_texts, val_labels, test_texts, test_labels,
    )
    dm.setup()
    num_batches = math.ceil(len(dm.train_dataset) / args.batch_size)
    steps_per_epoch = math.ceil(num_batches / args.grad_accum)
    total_steps = max(1, steps_per_epoch * args.epochs)

    model = ClarityClassifier(
        model_name=args.model,
        learning_rate=args.lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        dropout=args.dropout,
        label_smoothing=args.label_smoothing,
        class_weights=class_weights,
        total_steps=total_steps,
        use_peft=args.use_peft,
        lora_r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        lora_target_modules=args.lora_target_modules,
    )

    fold_dir = output_dir / f"fold-{fold_idx + 1}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    callbacks = [
        ModelCheckpoint(dirpath=fold_dir, filename="best-{epoch:02d}-{val/f1_macro:.4f}", monitor="val/f1_macro", mode="max", save_top_k=1, save_last=True),
        EarlyStopping(monitor="val/f1_macro", patience=args.patience, mode="max", min_delta=0.001),
        LearningRateMonitor(logging_interval="step"),
    ]
    logger = CSVLogger(save_dir=fold_dir, name="logs")
    trainer = pl.Trainer(
        accelerator=args.accelerator,
        devices=args.devices,
        max_epochs=args.epochs,
        callbacks=callbacks,
        logger=logger,
        enable_progress_bar=True,
        gradient_clip_val=1.0,
        accumulate_grad_batches=args.grad_accum,
        precision=args.precision,
        deterministic=True,
        log_every_n_steps=10,
    )
    trainer.fit(model, dm)
    if test_texts:
        trainer.test(model, dm, ckpt_path="best")

    final_dir = fold_dir / "final"
    final_dir.mkdir(exist_ok=True)
    best_model = ClarityClassifier.load_from_checkpoint(callbacks[0].best_model_path)
    best_model.model.save_pretrained(final_dir)
    dm.tokenizer.save_pretrained(final_dir)
    if args.use_peft:
        (final_dir / "peft_base_model.txt").write_text(args.model)
        if args.merge_lora and PeftModel:
            merge_dir = Path(args.merge_lora_output) if args.merge_lora_output else (fold_dir / "merged")
            merge_dir.mkdir(exist_ok=True)
            merged_model = best_model.model.merge_and_unload()
            merged_model.save_pretrained(merge_dir)
            dm.tokenizer.save_pretrained(merge_dir)
            (merge_dir / "merged_from_peft.txt").write_text(str(final_dir))

    if args.overwrite_ensemble_dirs:
        root = Path(args.overwrite_ensemble_root)
        target = root / f"roberta-ensemble-model-{fold_idx + 1}" / "final"
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(final_dir, target)
        if args.use_peft and args.merge_lora:
            ms = Path(args.merge_lora_output) if args.merge_lora_output else (fold_dir / "merged")
            mt = root / f"roberta-ensemble-model-{fold_idx + 1}" / "merged"
            if mt.exists():
                shutil.rmtree(mt)
            shutil.copytree(ms, mt)

    return {"fold": fold_idx + 1, "best_val_f1": float(callbacks[0].best_model_score), "best_model_path": str(final_dir)}


def train_ensemble(args):
    pl.seed_everything(args.seed, workers=True)
    train_texts, train_labels, test_texts, test_labels = load_qevasion_data()
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path(args.output_dir) / f"ensemble_{args.model.replace('/', '_')}_{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "config.json").write_text(json.dumps({**vars(args), "timestamp": timestamp, "train_size": len(train_texts), "test_size": len(test_texts)}, indent=2))
    skf = StratifiedKFold(n_splits=args.num_folds, shuffle=True, random_state=args.seed)
    results = []
    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(train_texts, train_labels)):
        fold_result = train_single_fold(
            fold_idx,
            [train_texts[i] for i in tr_idx], [train_labels[i] for i in tr_idx],
            [train_texts[i] for i in val_idx], [train_labels[i] for i in val_idx],
            test_texts, test_labels,
            args, output_dir,
        )
        results.append(fold_result)
    (output_dir / "ensemble_results.json").write_text(json.dumps(results, indent=2))
    print("\nENSEMBLE DONE. Avg val F1:", np.mean([r["best_val_f1"] for r in results]))
    return results

## 8. Run training

Uses `CONFIG` from cell 2. Re-run that cell to change options, then run this cell.

In [ ]:
args = SimpleNamespace(**CONFIG)
results = train_ensemble(args)
results